# Solving the 1D heat equation: PINN, spline PIKAN, and finite differences

This notebook implements the comparison proposed in `PIKAN_Heat_Equation_Academic_Project_Proposal.pdf` and adds a classical Crank–Nicolson finite difference solver. Both networks learn the same initial–boundary value problem using physics losses and **no interior temperature labels**. The exact solution is reserved for validation and the final evaluation.

**Study scope:** This is a controlled implementation exercise and architecture comparison for one smooth PDE instance, not a practical speed benchmark for general heat solvers. Because this particular case has a closed-form solution, direct formula evaluation is also benchmarked below.

Run top to bottom in the `ml` environment. The default experiment uses CPU so the timing comparison uses the same hardware for every run. For a faster draft, reduce `STEPS` and/or `SEEDS` below; rerun all cells before interpreting the conclusions.

**Problem:** $T_t=\alpha T_{xx}$ on $(x,t)\in[0,1]^2$, $\alpha=0.1$, $T(x,0)=\sin(\pi x)$, $T(0,t)=T(1,t)=0$. The exact solution is $T(x,t)=e^{-\alpha\pi^2t}\sin(\pi x)$.

In [1]:
import math
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cpu")
DTYPE = torch.float32
ALPHA = 0.1
STEPS = 400
SEEDS = (0, 1, 2, 3, 4)
N_INTERIOR, N_INITIAL, N_BOUNDARY = 96, 48, 48
LR = 3e-3
WEIGHTS = {"physics": 1.0, "initial": 10.0, "boundary": 10.0}
print(f"PyTorch {torch.__version__}; device={DEVICE}; steps={STEPS}; seeds={SEEDS}")

PyTorch 2.13.0+cu130; device=cpu; steps=400; seeds=(0, 1, 2, 3, 4)


## 1. Exact solution and collocation points

Each seed generates one fixed collection of uniformly sampled interior, initial, and boundary points. The MLP and KAN for that seed receive **identical points**, weights, Adam settings, and update counts. Fixed points make this small comparison easy to reproduce; they also mean the result measures fit to one collocation design per seed.

In [2]:
def exact_temperature(coords):
    x, t = coords[:, :1], coords[:, 1:2]
    return torch.exp(-ALPHA * math.pi**2 * t) * torch.sin(math.pi * x)


def sample_points(seed):
    rng = torch.Generator(device="cpu").manual_seed(seed)
    interior = torch.rand(N_INTERIOR, 2, generator=rng, dtype=DTYPE)
    initial = torch.cat((torch.rand(N_INITIAL, 1, generator=rng, dtype=DTYPE),
                         torch.zeros(N_INITIAL, 1, dtype=DTYPE)), dim=1)
    boundary = torch.rand(N_BOUNDARY, 2, generator=rng, dtype=DTYPE)
    boundary[:, 0] = torch.arange(N_BOUNDARY) % 2
    return interior.to(DEVICE), initial.to(DEVICE), boundary.to(DEVICE)


def pde_residual(model, coords, create_graph=True):
    coords = coords.detach().clone().requires_grad_(True)
    value = model(coords)
    first = torch.autograd.grad(value, coords, torch.ones_like(value), create_graph=True)[0]
    second_x = torch.autograd.grad(first[:, :1], coords,
                                   torch.ones_like(first[:, :1]),
                                   create_graph=create_graph)[0][:, :1]
    return first[:, 1:2] - ALPHA * second_x


class ExactModel(nn.Module):
    def forward(self, coords):
        return exact_temperature(coords)


probe = torch.rand(64, 2, generator=torch.Generator().manual_seed(2026))
exact_residual = pde_residual(ExactModel(), probe, create_graph=False)
assert exact_residual.abs().max() < 2e-6
print("Exact-solution PDE residual, max absolute:", f"{exact_residual.abs().max().item():.2e}")

Exact-solution PDE residual, max absolute: 1.19e-07
